# NLP

# Setup 
- **NLTK** - classic teaching toolkit (tokenizers, stemmers, corpora)
- **scikit-learn** - vectorizes + ML models
- **gensim** = Word2Vec training
- **spaCy** - industrial NER & dependency parsing

In [9]:
import nltk

packages = [
    'punkt',                # Tokenizers
    'stopwords',            # stopword lists
    'wordnet', 'omw-1.4',   # lemmatizer dictionary
    'averaged_perceptron_tagger', # POS tagger
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words', # NER chunker
    'vader_lexicon',        # sentiment lexicon
]

for pkg in packages:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f'Could not download {pkg} : {e}')

print('setup complete')

setup complete


# Tokenization

**Idea:** computers can't read a sentence as one blob - we break it into pieces called tokens (usually words, cometimes sentences or sub-words). Tokenization is always the first step of any NLP pipeline.

**Why it matters:** Every later step (counting words, taggin, classifying) operates on tokens. Bad tokenization = bad everything downstream

**Watch out for:** punctuation, contractions(`don't` -> `do` + `n't`) and abbreviations(U.S.A.)


In [20]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "Dr. Smith isn't happy. NLP is fun, and it's powerful! Visit openai.com today."

# Sentence tokenization - splits on sentece boundaries (note: it keeps 'Dr.' intact)
sentences = sent_tokenize(text)
print('SENTENCES:')
for i, s in enumerate(sentences, 1):
    print(f' {i}. {s}')

# Word tokenization - splits into words + punctuation
words = word_tokenize(text)
print('\nWORDS:')
for i, s in enumerate(words, 1):
    print(f' {i}. {s}')

print(f'\n{len(sentences)} sentences, {len(words)} word-tokens')

SENTENCES:
 1. Dr. Smith isn't happy.
 2. NLP is fun, and it's powerful!
 3. Visit openai.com today.

WORDS:
 1. Dr.
 2. Smith
 3. is
 4. n't
 5. happy
 6. .
 7. NLP
 8. is
 9. fun
 10. ,
 11. and
 12. it
 13. 's
 14. powerful
 15. !
 16. Visit
 17. openai.com
 18. today
 19. .

3 sentences, 19 word-tokens


# Stemming

**Idea:** Reduce a word to its root by chopping off endings. `running`, `runs`, `ran`(no) -> `run`. It uses crude rules, so the output is not always a real word (`studies` -> `studi`)

**Trade-off:** Vey fast, but sloppy. Good enough for search engines and rough text matching

The **Porter stemmer** is the classic, **Snowball** is a slightly improved version.

In [23]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer('english')

words = ['running', 'runs', 'runner', 'easily', 'studies', 'studying', 'happily', 'connection']

print(f"{'WORD':<12}{'PORTER':<12}{'SNOWBALL':<12}")

print('-' * 36)

for w in words:
    print(f"{w:<12}{porter.stem(w):<12}{snowball.stem(w):<12}")

WORD        PORTER      SNOWBALL    
------------------------------------
running     run         run         
runs        run         run         
runner      runner      runner      
easily      easili      easili      
studies     studi       studi       
studying    studi       studi       
happily     happili     happili     
connection  connect     connect     


# 3. Lemmatization

**Idea:** Like stemming, but smart. It uses a dictionary (WordNet) and the word's part-of-speech to return the real base word (the lemma)

`studies` -> `study`, `better` -> `good`, `mice` -> `mouse`

**Trade-off:** Slower and needs a POS hint, but the output is always a valid word. Preferred when accuracy matters.

**Key Gotcha:** By default the lemmatizer assumes every word is a noun. Give it the correct POS tag(`v` for verb, `a` for adjective) and results improve dramatiaclly.

In [27]:
from nltk.stem import WordNetLemmatizer

lemma = WordNetLemmatizer()

print('WITHOUT POS (treated as noun):')
for w in ['running', 'studies', 'better', 'mice', 'was']:
    print(f' {w:<10} -> {lemma.lemmatize(w)}')

print('\nWITH correct POS:')
print(f" running (verb) -> {lemma.lemmatize('running', pos='v')}")
print(f" studies (verb) -> {lemma.lemmatize('studies', pos='v')}")
print(f" better  (adj)  -> {lemma.lemmatize('better', pos='a')}")
print(f" was     (verb) -> {lemma.lemmatize('was', pos='v')}")

print('\nStemming vs Lemmatization on "studies":')

print(f" Porter stem -> {porter.stem('studies')} (not a real word)")
print(f" Lemma (word) -> {lemma.lemmatize('studies', pos='v')}")

WITHOUT POS (treated as noun):
 running    -> running
 studies    -> study
 better     -> better
 mice       -> mouse
 was        -> wa

WITH correct POS:
 running (verb) -> run
 studies (verb) -> study
 better  (adj)  -> good
 was     (verb) -> be

Stemming vs Lemmatization on "studies":
 Porter stem -> studi (not a real word)
 Lemma (word) -> study


# 